In [4]:
# bow vs tfidf

# Import necessary libraries
import mlflow
import mlflow.sklearn
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import pandas as pd
import re
import string
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import numpy as np
import os


In [16]:
# Set MLflow tracking URI to DagsHub
mlflow.set_tracking_uri("https://dagshub.com/DeepuML/Mlops-Mini-Project.mlflow")

In [8]:
# Load the data
df = pd.read_csv('https://raw.githubusercontent.com/campusx-official/jupyter-masterclass/main/tweet_emotions.csv').drop(columns=['tweet_id'])
df.head()


,sentiment,content
0,empty,@tiffanylue i know i was listenin to bad habi...
1,sadness,Layin n bed with a headache ughhhh...waitin o...
2,sadness,Funeral ceremony...gloomy friday...
3,enthusiasm,wants to hang out with friends SOON!
4,neutral,@dannycastillo We want to trade with someone w...


In [9]:
# Define text preprocessing functions
def lemmatization(text):
    """Lemmatize the text."""
    lemmatizer = WordNetLemmatizer()
    text = text.split()
    text = [lemmatizer.lemmatize(word) for word in text]
    return " ".join(text)

def remove_stop_words(text):
    """Remove stop words from the text."""
    stop_words = set(stopwords.words("english"))
    text = [word for word in str(text).split() if word not in stop_words]
    return " ".join(text)

def removing_numbers(text):
    """Remove numbers from the text."""
    text = ''.join([char for char in text if not char.isdigit()])
    return text

def lower_case(text):
    """Convert text to lower case."""
    text = text.split()
    text = [word.lower() for word in text]
    return " ".join(text)

def removing_punctuations(text):
    """Remove punctuations from the text."""
    text = re.sub('[%s]' % re.escape(string.punctuation), ' ', text)
    text = text.replace('؛', "")
    text = re.sub('\s+', ' ', text).strip()
    return text

def removing_urls(text):
    """Remove URLs from the text."""
    url_pattern = re.compile(r'https?://\S+|www\.\S+')
    return url_pattern.sub(r'', text)

def normalize_text(df):
    """Normalize the text data."""
    try:
        df['content'] = df['content'].apply(lower_case)
        df['content'] = df['content'].apply(remove_stop_words)
        df['content'] = df['content'].apply(removing_numbers)
        df['content'] = df['content'].apply(removing_punctuations)
        df['content'] = df['content'].apply(removing_urls)
        df['content'] = df['content'].apply(lemmatization)
        return df
    except Exception as e:
        print(f'Error during text normalization: {e}')
        raise


<>:30: SyntaxWarning: invalid escape sequence '\s'
<>:30: SyntaxWarning: invalid escape sequence '\s'
C:\Users\Deepu\AppData\Local\Temp\ipykernel_21960\720468952.py:30: SyntaxWarning: invalid escape sequence '\s'
  text = re.sub('\s+', ' ', text).strip()


In [10]:
# Normalize the text data
df = normalize_text(df)

x = df['sentiment'].isin(['happiness','sadness'])
df = df[x]

df['sentiment'] = df['sentiment'].replace({'sadness':0, 'happiness':1})


C:\Users\Deepu\AppData\Local\Temp\ipykernel_21960\3352374845.py:7: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['sentiment'] = df['sentiment'].replace({'sadness':0, 'happiness':1})


In [11]:
# Set the experiment name
mlflow.set_experiment("Bow vs TfIdf")

2025/09/28 00:45:45 INFO mlflow.tracking.fluent: Experiment with name 'Bow vs TfIdf' does not exist. Creating a new experiment.


<Experiment: artifact_location='mlflow-artifacts:/69093d60ee2c40da8a6323e893b7f444', creation_time=1759000545742, experiment_id='1', last_update_time=1759000545742, lifecycle_stage='active', name='Bow vs TfIdf', tags={}>

In [12]:
# Define feature extraction methods
vectorizers = {
    'BoW': CountVectorizer(),
    'TF-IDF': TfidfVectorizer()
}

In [13]:
# Define algorithms
algorithms = {
    'LogisticRegression': LogisticRegression(),
    'MultinomialNB': MultinomialNB(),
    'XGBoost': XGBClassifier(),
    'RandomForest': RandomForestClassifier(),
    'GradientBoosting': GradientBoostingClassifier()
}


In [17]:
import pickle
import tempfile
import warnings
from urllib3.exceptions import InsecureRequestWarning

# Suppress warnings and MLflow UI messages
warnings.filterwarnings('ignore', category=InsecureRequestWarning)
os.environ["MLFLOW_TRACKING_INSECURE_TLS"] = "true"

# Start the parent run
with mlflow.start_run(run_name="All Experiments") as parent_run:
    # Loop through algorithms and feature extraction methods (Child Runs)
    for algo_name, algorithm in algorithms.items():
        for vec_name, vectorizer in vectorizers.items():
            with mlflow.start_run(run_name=f"{algo_name} with {vec_name}", nested=True) as child_run:
                X = vectorizer.fit_transform(df['content'])
                y = df['sentiment']
                X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
                

                # Log preprocessing parameters
                mlflow.log_param("vectorizer", vec_name)
                mlflow.log_param("algorithm", algo_name)
                mlflow.log_param("test_size", 0.2)
                
                # Model training
                model = algorithm
                model.fit(X_train, y_train)
                
                # Log model parameters
                if algo_name == 'LogisticRegression':
                    mlflow.log_param("C", model.C)
                elif algo_name == 'MultinomialNB':
                    mlflow.log_param("alpha", model.alpha)
                elif algo_name == 'XGBoost':
                    mlflow.log_param("n_estimators", model.n_estimators)
                    mlflow.log_param("learning_rate", model.learning_rate)
                elif algo_name == 'RandomForest':
                    mlflow.log_param("n_estimators", model.n_estimators)
                    mlflow.log_param("max_depth", model.max_depth)
                elif algo_name == 'GradientBoosting':
                    mlflow.log_param("n_estimators", model.n_estimators)
                    mlflow.log_param("learning_rate", model.learning_rate)
                    mlflow.log_param("max_depth", model.max_depth)
                
                # Model evaluation
                y_pred = model.predict(X_test)
                accuracy = accuracy_score(y_test, y_pred)
                precision = precision_score(y_test, y_pred)
                recall = recall_score(y_test, y_pred)
                f1 = f1_score(y_test, y_pred)
                
                # Log evaluation metrics
                mlflow.log_metric("accuracy", accuracy)
                mlflow.log_metric("precision", precision)
                mlflow.log_metric("recall", recall)
                mlflow.log_metric("f1_score", f1)
                
                # Save model manually and log as artifact for DagsHub compatibility
                with tempfile.NamedTemporaryFile(suffix='.pkl', delete=False) as f:
                    pickle.dump(model, f)
                    model_path = f.name
                
                mlflow.log_artifact(model_path, "model")
                
                # Also save vectorizer for future use
                with tempfile.NamedTemporaryFile(suffix='.pkl', delete=False) as f:
                    pickle.dump(vectorizer, f)
                    vectorizer_path = f.name
                
                mlflow.log_artifact(vectorizer_path, "model")
                
                # Clean up temporary files
                os.unlink(model_path)
                os.unlink(vectorizer_path)
                
                # Print the results for verification
                print("="*60)
                print(f"🔬 Algorithm: {algo_name} | Feature Engineering: {vec_name}")
                print("="*60)
                print(f"Accuracy:  {accuracy:.4f}")
                print(f"Precision: {precision:.4f}")
                print(f"Recall:    {recall:.4f}")
                print(f"F1 Score:  {f1:.4f}")
                print("="*60)

🔬 Algorithm: LogisticRegression | Feature Engineering: BoW
Accuracy:  0.7937
Precision: 0.7847
Recall:    0.7970
F1 Score:  0.7908
🏃 View run LogisticRegression with BoW at: https://dagshub.com/DeepuML/Mlops-Mini-Project.mlflow/#/experiments/1/runs/ddc7787dcc5b456c8b04b2119d3e1c19
🧪 View experiment at: https://dagshub.com/DeepuML/Mlops-Mini-Project.mlflow/#/experiments/1
🏃 View run LogisticRegression with BoW at: https://dagshub.com/DeepuML/Mlops-Mini-Project.mlflow/#/experiments/1/runs/ddc7787dcc5b456c8b04b2119d3e1c19
🧪 View experiment at: https://dagshub.com/DeepuML/Mlops-Mini-Project.mlflow/#/experiments/1
🔬 Algorithm: LogisticRegression | Feature Engineering: TF-IDF
Accuracy:  0.7942
Precision: 0.7779
Recall:    0.8108
F1 Score:  0.7940
🔬 Algorithm: LogisticRegression | Feature Engineering: TF-IDF
Accuracy:  0.7942
Precision: 0.7779
Recall:    0.8108
F1 Score:  0.7940
🏃 View run LogisticRegression with TF-IDF at: https://dagshub.com/DeepuML/Mlops-Mini-Project.mlflow/#/experiments/1